# 5 — Within-model disagreement benchmark

Compare disjoint two-run and three-run prescribed labels within each model,
averaging all ten partitions. Between models, compare all 100 subset combinations
in both directions for each pair, matching aggregation sizes. Ties select the
modal category nearest the subset median, then the lower category.

Fixed prompts are primary; randomized batches are a separate sensitivity
analysis that also includes batch-context variation. Each description has equal
weight. Overlapping subset comparisons are not independent replications.

Inputs: `intermediate/s1_labels.pkl`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from itertools import combinations

In [2]:
INTERMEDIATE = Path("intermediate")
classified = pd.read_pickle(INTERMEDIATE / "s1_labels.pkl")

original_cube = classified["original_cube"]
fixed_cube = classified["fixed_cube"]
fixed_ids = classified["fixed_ids"]
MODEL_ORDER = classified["model_order"]
LABEL_NAMES = classified["label_names"]
RUNS = classified["runs"]

In [3]:
# Match the original benchmark ordering before drawing bootstrap samples.
fixed_order = np.argsort(fixed_ids)
cubes = {
    "Fixed prompt (primary)": fixed_cube[fixed_order],
    "Randomized batches (sensitivity)": original_cube,
}

In [4]:
LEVELS = np.arange(1, 6)
TWO_RUN_SUBSETS = list(combinations(range(5), 2))
THREE_RUN_SUBSETS = [
    tuple(r for r in range(5) if r not in two) for two in TWO_RUN_SUBSETS
]
MODEL_PAIRS = list(combinations(range(6), 2))


def modal_label(values):
    """Reduce the last (run) axis with the mode → median proximity → lower rule."""
    counts = (values[..., :, None] == LEVELS).sum(axis=-2)
    tied_modes = counts == counts.max(axis=-1, keepdims=True)
    distance = np.abs(LEVELS - np.median(values, axis=-1)[..., None])
    # argmin chooses the first/lower category if distances still tie.
    return np.where(tied_modes, distance, np.inf).argmin(axis=-1) + 1


def metrics(a, b):
    """Rows are descriptions; columns are subset comparisons. Macro-average kappa."""
    a, b = a.astype(float), b.astype(float)
    difference = a - b
    observed = np.mean(difference**2, axis=0)
    expected = (
        np.mean(a * a, axis=0)
        + np.mean(b * b, axis=0)
        - 2 * a.mean(axis=0) * b.mean(axis=0)
    )
    # The common /16 quadratic weighting factor cancels in observed/expected.
    return {
        "agreement": float(np.mean(a == b)),
        "disagreement": float(np.mean(a != b)),
        "ordinal_distance": float(np.mean(np.abs(difference))),
        "weighted_kappa": float(np.mean(1 - observed / expected)),
    }


summary_rows, within_rows, between_rows, partition_rows = [], [], [], []
description_disagreement = {}
for experiment, cube in cubes.items():
    two = np.stack(
        [modal_label(cube[..., subset]) for subset in TWO_RUN_SUBSETS], axis=-1
    )
    three = np.stack(
        [modal_label(cube[..., subset]) for subset in THREE_RUN_SUBSETS], axis=-1
    )
    # shape: descriptions × models × ten subsets, with complementary subsets aligned.
    for m, model in enumerate(MODEL_ORDER):
        within_rows.append(
            {
                "experiment": experiment,
                "model": model,
                **metrics(two[:, m, :], three[:, m, :]),
            }
        )
    for split, (subset2, subset3) in enumerate(zip(TWO_RUN_SUBSETS, THREE_RUN_SUBSETS)):
        partition_rows.append(
            {
                "experiment": experiment,
                "two_runs": tuple(RUNS[i] for i in subset2),
                "three_runs": tuple(RUNS[i] for i in subset3),
                "disagreement": float(np.mean(two[:, :, split] != three[:, :, split])),
            }
        )
    description_disagreement[(experiment, "Within model (benchmark)")] = (
        two != three
    ).mean(axis=(1, 2))
    between_description_vectors = []
    for a, b in MODEL_PAIRS:
        # Independent subset choices across models; include both 2-versus-3 directions.
        first_a, first_b = np.broadcast_arrays(two[:, a, :, None], three[:, b, None, :])
        second_a, second_b = np.broadcast_arrays(
            two[:, b, :, None], three[:, a, None, :]
        )
        left = np.concatenate(
            [first_a.reshape(len(cube), 100), second_a.reshape(len(cube), 100)], axis=1
        )
        right = np.concatenate(
            [first_b.reshape(len(cube), 100), second_b.reshape(len(cube), 100)], axis=1
        )
        between_description_vectors.append((left != right).mean(axis=1))
        between_rows.append(
            {
                "experiment": experiment,
                "model_a": MODEL_ORDER[a],
                "model_b": MODEL_ORDER[b],
                **metrics(left, right),
            }
        )
    description_disagreement[(experiment, "Between models")] = np.mean(
        between_description_vectors, axis=0
    )
    for scope, rows in [
        ("Within model (benchmark)", within_rows),
        ("Between models", between_rows),
    ]:
        selected = [row for row in rows if row["experiment"] == experiment]
        summary_rows.append(
            {
                "experiment": experiment,
                "comparison": scope,
                "descriptions": len(cube),
                **{
                    metric: float(np.mean([row[metric] for row in selected]))
                    for metric in [
                        "agreement",
                        "disagreement",
                        "ordinal_distance",
                        "weighted_kappa",
                    ]
                },
            }
        )

within_details = pd.DataFrame(within_rows)
between_details = pd.DataFrame(between_rows)
partition_details = pd.DataFrame(partition_rows)
comparison = pd.DataFrame(summary_rows)
print(
    "Complete: all ten within-model partitions and all 200 subset comparisons per model pair."
)

# Shared description resamples pair the within- and between-model reference quantities.
benchmark_rng = np.random.default_rng(20261107)
ci_rows = []
for experiment, cube in cubes.items():
    indices = benchmark_rng.integers(0, len(cube), size=(1000, len(cube)))
    for scope in ["Within model (benchmark)", "Between models"]:
        vector = description_disagreement[(experiment, scope)]
        observed = comparison.loc[
            comparison.experiment.eq(experiment) & comparison.comparison.eq(scope),
            "disagreement",
        ].iloc[0]
        lo, hi = np.quantile(vector[indices].mean(axis=1), [0.025, 0.975])
        ci_rows.append(
            {
                "experiment": experiment,
                "comparison": scope,
                "disagreement_ci_low": lo,
                "disagreement_ci_high": hi,
            }
        )
comparison = comparison.merge(
    pd.DataFrame(ci_rows), on=["experiment", "comparison"], validate="one_to_one"
)

Complete: all ten within-model partitions and all 200 subset comparisons per model pair.


In [5]:
display_table = pd.DataFrame(
    {
        "Experiment": comparison.experiment,
        "Comparison": comparison.comparison,
        "Weighted kappa": comparison.weighted_kappa,
        "Exact disagreement (95% CI)": [
            f"{r.disagreement:.2%} ({r.disagreement_ci_low:.2%}–{r.disagreement_ci_high:.2%})"
            for r in comparison.itertuples(index=False)
        ],
        "Mean ordinal distance": comparison.ordinal_distance,
    }
)
display(
    display_table.style.hide(axis="index").format(
        {"Weighted kappa": "{:.3f}", "Mean ordinal distance": "{:.3f}"}
    )
)

Experiment,Comparison,Weighted kappa,Exact disagreement (95% CI),Mean ordinal distance
Fixed prompt (primary),Within model (benchmark),0.919,8.78% (8.18%–9.39%),0.089
Fixed prompt (primary),Between models,0.701,33.64% (32.37%–34.96%),0.344
Randomized batches (sensitivity),Within model (benchmark),0.752,25.74% (25.28%–26.19%),0.260
Randomized batches (sensitivity),Between models,0.682,34.71% (34.18%–35.19%),0.355


Kappa is calculated per subset comparison and then averaged. Disagreement
intervals use 1,000 paired description resamples conditional on the five runs.
This supplies a within-task reference, not human ground truth or evidence
that disagreement is unique to LLMs.

## Save results

In [6]:
pd.to_pickle(
    {
        "comparison": comparison,
        "within_details": within_details,
        "between_details": between_details,
    },
    INTERMEDIATE / "s5_benchmark.pkl",
)